# SU(4) exact fourth-order completion — clean no-edit reproduction

Run the code cells once, in order. No source patching or path editing is required.

The notebook performs only the certified chain:

1. verify and extract the complete release;
2. create an isolated Python environment with the pinned dependencies;
3. recompute all 35,130 balanced paths and 1,806 exceptional paths into a fresh directory;
4. run the exact independent artifact verifier;
5. compare the regenerated kernel byte-for-byte with the frozen normative kernel;
6. package the regenerated results, logs, and manifest.

It intentionally makes no claim of a separate 30-topology cross-engine regression.

In [ ]:
# 1. Locate/upload, integrity-check, and extract the clean release
from __future__ import annotations
import hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
BASE = Path('/content') if Path('/content').exists() else Path('/mnt/data')
WORK = BASE / 'SU4_CLEAN_NO_EDIT_RUN'
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def candidates() -> list[Path]:
    names = (
        'SU4_DETERMINANT_COMPLETE_RELEASE_V2_CLEAN*.zip',
        'SU4_DETERMINANT_COMPLETE_RELEASE*.zip',
    )
    out = []
    for root in (BASE, Path('/mnt/data'), Path.cwd()):
        if root.exists():
            for pattern in names:
                out.extend(root.glob(pattern))
    return sorted(set(p.resolve() for p in out), key=lambda p: (0 if 'V2_CLEAN' in p.name else 1, str(p)))

found = candidates()
if not found and IN_COLAB:
    from google.colab import files
    print('Upload SU4_DETERMINANT_COMPLETE_RELEASE_V2_CLEAN.zip')
    uploaded = files.upload()
    found = [BASE / Path(name).name for name in uploaded]
if not found:
    raise FileNotFoundError('SU(4) clean release ZIP was not found.')

RELEASE_ZIP = found[0]
with zipfile.ZipFile(RELEASE_ZIP) as zf:
    if zf.testzip() is not None:
        raise RuntimeError('ZIP integrity failure')
    for info in zf.infolist():
        target = (WORK / info.filename).resolve()
        if not str(target).startswith(str(WORK.resolve()) + os.sep):
            raise RuntimeError(f'unsafe ZIP member: {info.filename}')
    zf.extractall(WORK)

roots = [p for p in WORK.iterdir() if p.is_dir() and (p / 'y4_su4_complete.py').is_file()]
if len(roots) != 1:
    raise RuntimeError(f'Expected one extracted release root, found {roots}')
ROOT = roots[0]

manifest = ROOT / 'RELEASE_SHA256SUMS.txt'
if not manifest.is_file():
    raise FileNotFoundError(manifest)
for line in manifest.read_text().splitlines():
    if not line.strip():
        continue
    expected, rel = line.split(None, 1)
    rel = rel.strip()
    if rel.startswith('./'):
        rel = rel[2:]
    path = ROOT / rel
    if sha256(path) != expected:
        raise RuntimeError(f'manifest mismatch: {rel}')

print('PASS ZIP integrity')
print('PASS internal SHA-256 manifest')
print('RELEASE', RELEASE_ZIP)
print('RELEASE SHA-256', sha256(RELEASE_ZIP))
print('ROOT', ROOT)

In [ ]:
# 2. Create an isolated environment with the pinned dependencies
VENV = WORK / '.venv'
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
PYTHON = VENV / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')

check = subprocess.run(
    [str(PYTHON), '-c', 'import numpy,sympy; print(numpy.__version__,sympy.__version__)'],
    text=True, capture_output=True
)
required = {'numpy': '2.3.5', 'sympy': '1.14.0'}
versions_ok = check.returncode == 0 and check.stdout.strip().split() == [required['numpy'], required['sympy']]
if not versions_ok:
    subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(ROOT/'NOTE_MISC_requirements.txt')], check=True)

versions = subprocess.check_output(
    [str(PYTHON), '-c', 'import numpy,sympy; print(numpy.__version__,sympy.__version__)'],
    text=True
).strip()
assert versions == '2.3.5 1.14.0', versions
print('PASS isolated dependency versions:', versions)

In [ ]:
# 3. Cold run into a fresh output directory
OUT = WORK / 'reproduced_results'
LOG = WORK / 'SU4_CLEAN_REPRODUCTION.log'
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()

command = [
    str(PYTHON), str(ROOT/'y4_su4_complete.py'),
    '--base-dir', str(ROOT),
    '--output-dir', str(OUT),
]
with LOG.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
    rc = process.wait()
if rc != 0:
    raise RuntimeError(f'complete contraction failed with exit code {rc}')
print('PASS clean complete contraction')

In [ ]:
# 4. Independent artifact verification and deterministic comparison
subprocess.run([
    str(PYTHON), str(ROOT/'verify_su4_complete.py'),
    '--artifact-dir', str(OUT),
], check=True)
subprocess.run([
    str(PYTHON), str(ROOT/'compare_reproduced.py'),
    '--reference', str(ROOT/'results'),
    '--regenerated', str(OUT),
], check=True)

cert = json.loads((OUT/'SU4_DETERMINANT_COMPLETE_CERTIFICATE.json').read_text())
assert cert['gates']['passed'] is True
assert cert['counts']['balanced_fusion_paths'] == 35130
assert cert['counts']['exceptional_fusion_paths'] == 1806
assert cert['counts']['exceptional_orbits'] == 156
assert cert['counts']['full_kernel_records'] == 189
assert cert['kernel']['semantic_sha256'] == 'e8bc5badc026d9874af56cd9ded47c4f0b0833597cb13ae5fd5e618a113dfeb9'
print(json.dumps(cert['coefficients'], indent=2))
print('ALL CLEAN NO-EDIT REPRODUCIBILITY GATES PASS')

In [ ]:
# 5. Freeze regenerated artifacts, logs, environment, and SHA-256 manifest
import platform, time
BUNDLE_DIR = WORK / 'SU4_CLEAN_NO_EDIT_REPRODUCED'
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir()
shutil.copytree(OUT, BUNDLE_DIR/'results')
shutil.copy2(LOG, BUNDLE_DIR/LOG.name)
for name in ('y4_su4_complete.py','verify_su4_complete.py','compare_reproduced.py','NOTE_MISC_requirements.txt','README.md'):
    shutil.copy2(ROOT/name, BUNDLE_DIR/name)

metadata = {
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'python': subprocess.check_output([str(PYTHON), '--version'], text=True).strip(),
    'platform': platform.platform(),
    'release_zip': RELEASE_ZIP.name,
    'release_zip_sha256': sha256(RELEASE_ZIP),
    'kernel_semantic_sha256': cert['kernel']['semantic_sha256'],
    'balanced_fusion_paths': 35130,
    'exceptional_fusion_paths': 1806,
}
(BUNDLE_DIR/'RUN_METADATA.json').write_text(json.dumps(metadata, indent=2, sort_keys=True)+'\n')

lines = []
for path in sorted(p for p in BUNDLE_DIR.rglob('*') if p.is_file() and p.name != 'SHA256SUMS.txt'):
    lines.append(f'{sha256(path)}  {path.relative_to(BUNDLE_DIR)}')
(BUNDLE_DIR/'SHA256SUMS.txt').write_text('\n'.join(lines)+'\n')

ARCHIVE = BASE / 'SU4_CLEAN_NO_EDIT_REPRODUCED.zip'
if ARCHIVE.exists():
    ARCHIVE.unlink()
with zipfile.ZipFile(ARCHIVE, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(p for p in BUNDLE_DIR.rglob('*') if p.is_file()):
        zf.write(path, arcname=f'{BUNDLE_DIR.name}/{path.relative_to(BUNDLE_DIR)}')
with zipfile.ZipFile(ARCHIVE) as zf:
    assert zf.testzip() is None
print('PASS output ZIP integrity')
print('OUTPUT', ARCHIVE)
print('OUTPUT SHA-256', sha256(ARCHIVE))

if IN_COLAB:
    from google.colab import files
    files.download(str(ARCHIVE))